In [1]:
import pandas as pd

df = pd.read_parquet('./youtube_trends_KR.parquet')

print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df['collection_date'].min(), '~', df['collection_date'].max())
print(f"고유 영상: {df['video_id'].nunique()}")
print(f"평균 등장 횟수: {len(df)/df['video_id'].nunique():.1f}회")

FileNotFoundError: [Errno 2] No such file or directory: './youtube_trends_KR.parquet'

In [ ]:
df['collection_date'] = pd.to_datetime(df['collection_date'])
df['published_at'] = pd.to_datetime(df['published_at'])

In [ ]:
# 지금 저장된 KR 데이터로 확인
import pandas as pd

df_kr = pd.read_csv('./data/youtube_kr.csv')

print("=== 기간 확인 ===")
print(df_kr['video_trending__date'].min())
print(df_kr['video_trending__date'].max())

print("\n=== Shorts 비율 확인 ===")
# PT 이후 M 앞 숫자 추출
import re

def get_duration_sec(duration):
    try:
        h = re.search(r'(\d+)H', duration)
        m = re.search(r'(\d+)M', duration)
        s = re.search(r'(\d+)S', duration)
        total = 0
        if h: total += int(h.group(1)) * 3600
        if m: total += int(m.group(1)) * 60
        if s: total += int(s.group(1))
        return total
    except:
        return None

df_kr['duration_sec'] = df_kr['video_duration'].apply(get_duration_sec)

shorts = df_kr[df_kr['duration_sec'] <= 60]
print(f"전체: {len(df_kr)}행")
print(f"Shorts(60초 이하): {len(shorts)}행")
print(f"Shorts 비율: {len(shorts)/len(df_kr)*100:.1f}%")

C:\Users\Playdata\AppData\Local\Temp\ipykernel_35092\443290564.py:4: DtypeWarning: Columns (24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_kr = pd.read_csv('./data/youtube_kr.csv')


=== 기간 확인 ===
2024.10.12
2026.04.26

=== Shorts 비율 확인 ===
전체: 111773행
Shorts(60초 이하): 1441행
Shorts 비율: 1.3%


In [ ]:
# 영상별 첫 등장일, 마지막 등장일
trend_life = df.groupby('video_id').agg(
    first_date=('collection_date', 'min'),
    last_date=('collection_date', 'max'),
    total_appearances=('collection_date', 'count'),
    max_rank=('rank', 'min'),  # 순위는 낮을수록 좋으니까 min
    max_views=('view_count', 'max')
).reset_index()

# 수명 계산
trend_life['lifespan_days'] = (
    trend_life['last_date'] - trend_life['first_date']
).dt.days + 1

In [ ]:
def label_stage(row, current_date):
    elapsed = (current_date - row['first_date']).days
    lifespan = row['lifespan_days']
    ratio = elapsed / lifespan if lifespan > 0 else 0
    
    if ratio <= 0.25:
        return 'early'
    elif ratio <= 0.50:
        return 'growing'
    elif ratio <= 0.75:
        return 'peak'
    else:
        return 'declining'

In [ ]:
# 수명 분포 확인
print(trend_life['lifespan_days'].describe())

# 상위 1% 제거 (극단적 이상치)
upper = trend_life['lifespan_days'].quantile(0.99)
trend_life = trend_life[trend_life['lifespan_days'] <= upper]